# Setup

In [ ]:
# Install pychometrics directly from GitHub
!pip install git+https://github.com/childmindresearch/llm_tracker

## ⚠️ Important: Data Privacy and Anonymization

Before using this tutorial with your own data, you should ensure that any sensitive or personally identifiable information (PII) has been removed or anonymized.

Because this package sends text to an LLM API for analysis, **any raw text you provide may leave your local environment** depending on your configuration. For this reason, you should **not upload identifiable or sensitive data unless it has been properly anonymized first**.

### A note on API data retention

Most companies will not use data you submit via an API (like in this tutorial) for training, and will delete the data after processing it within 30 days (e.g., OpenAI's policy: https://developers.openai.com/api/docs/guides/your-data). Policies vary by provider — check the terms of the model you select on OpenRouter — but in general, API submissions are treated more privately than chat-product submissions. This does **not** remove your responsibility to anonymize sensitive data first; it just means that, when working with properly anonymized text, API analysis is generally safe.

### Recommended approaches

If you are working with real data, we strongly recommend anonymizing it locally before running the analysis. We recommend:

- https://github.com/microsoft/presidio
- https://github.com/childmindresearch/anonymize-pii

These tools can help remove names, locations, emails, and other identifying information before any LLM processing occurs.

### If you prefer not to use real data

If you are not comfortable anonymizing your own dataset, you can use the **publicly available Reddit data provided in this repository**, which is safe to use for demonstration purposes.

---

**In short:**
✔ Use anonymized data whenever possible
✔ Be aware of your provider's API data retention policy
✔ When in doubt, use the provided sample dataset

# LLM Tracker Tutorial — V3

**LLM Tracker** is a Python package for identifying instances of psychological constructs in qualitative text data — such as clinical documents, therapy transcripts, or open-ended survey responses — using large language models (LLMs).

The core workflow has four steps:

1. **LLM Coding** — run a language model over a set of documents to identify and extract quotes that are instances of psychological constructs defined in a codebook.
2. **Human Coding** — produce a second set of codings to compare against. In production this is real human-coded data; for testing purposes a second LLM run can serve as a stand-in.
3. **Comparison** — align the two sets of codings instance-by-instance using an LLM matcher, classifying each as a true positive, false positive, or false negative.
4. **Summary Tables** — compute pooled, per-document, and weighted performance metrics across constructs.

This V3 tutorial is split into two parts:

- **Part 1 — Dummy Data Workflow:** runs the LLM twice over the same documents, using the second run as a stand-in for human coding. Useful for end-to-end pipeline testing without needing real human data.
- **Part 2 — Real Human Data Workflow:** the typical research workflow, where the LLM is run once and compared against externally produced human-coded data loaded from a CSV/xlsx file.

**Prerequisites:**
- An [OpenRouter](https://openrouter.ai) account and API key. OpenRouter provides access to a wide range of models (GPT-4o, Gemini, Claude, Llama, etc.) through a single API. Create an account, add credits, and generate a key at https://openrouter.ai/settings/keys.
- A directory of document files (`.txt`) **or** a CSV with one document per row.
- A JSON codebook defining the psychological constructs to identify (see `suicide_risk_codebook.json` for an example).

## Imports

Import the core classes and functions used throughout this tutorial:
- `PychometricsAnalyzer` — runs LLM coding over a directory of document files or a CSV
- `PychometricsComparer` — aligns and compares two sets of codings (human vs LLM)
- `format_comparison_table` — converts raw comparison output into a row-level DataFrame
- `compute_summary_tables` — computes per-document, concatenated, and weighted summary metrics
- `format_concatenated`, `format_per_interview`, `format_weighted_summary` — pretty-print the three summary tables
- `load_human_coding` — load human-coded data from a CSV/xlsx (Dedoose-compatible defaults)
- `validate_against_codebook` — sanity-check that human-coded constructs all match the codebook

In [ ]:
import pandas as pd
from llm_tracker import PychometricsAnalyzer
from llm_tracker.comparison import (
    PychometricsComparer,
    compute_summary_tables,
    format_comparison_table,
    format_concatenated,
    format_per_interview,
    format_weighted_summary,
)
from llm_tracker.file_handlers import load_human_coding, validate_against_codebook
from llm_tracker.utils import format_coding_table

## Paths and Configuration

Set all paths and credentials here before running any other cells. This is the only cell you need to edit before testing.

- `input_dir` — directory containing your `.txt` document files (used only for the `interview` data type)
- `csv_path` — CSV file with one document per row (used only for the `csv` data type)
- `codebook_path` — path to your JSON codebook. This defines the psychological constructs the LLM will look for and extract quotes for
- `api_key` — your OpenRouter API key, **or** the path to a `.env` file containing it. **Important:** OpenRouter API keys are case-sensitive — copy the key exactly as shown in your OpenRouter dashboard
- `llm_model` — the model to use for both coding and comparison. Any model listed at https://openrouter.ai/models can be passed here. You can use different models for coding vs comparison if desired (see Part 2's Comparison step)

In [ ]:
# Fill these in before running. All left blank by default.
api_key       = ""
llm_model     = ""

# For the dummy / Part-1 LLM coding (CSV-style data):
csv_path      = ""
text_column   = ""
codebook_path = ""

# Optional CSV-only metadata columns (used to build the document_id):
subreddit_column = ""   # e.g. "subreddit"; leave "" if not applicable
author_column    = ""   # e.g. "author";    leave "" if not applicable

# For the .txt directory style (alternative to CSV):
input_dir     = ""

# For Part 2 (real human data):
human_csv_path = ""   # path to the CSV/xlsx of human-coded excerpts

---

# Part 1 — Dummy Data Workflow

In this part we run the LLM twice over the same documents and treat the second run as a stand-in for human coding. This lets you test the full pipeline end-to-end before you have real human-coded data to compare against.

Because the same model is run twice on the same documents, agreement will be high but not perfect — the LLM introduces some variability between runs, which makes this a reasonable functional test of the whole workflow.

## Step 1 (Dummy): LLM Coding

This step runs the LLM over every document and extracts quotes that are instances of each construct defined in the codebook. Each document is processed independently, and results are saved to a timestamped output directory so runs are never overwritten.

The output directory has the following structure:
```
LLM_coding_YYYY-MM-DD_HHMMSS/
    codings/     <- one JSON per document, containing extracted construct instances
    metadata/    <- token usage, latency, and model info per document
    errors/      <- any documents that failed, with error messages
    README.md
```

Each coding JSON contains a list of instances, where each instance includes the construct name, the extracted quote, character-level indices into the source text, and a confidence score from the LLM.

In [ ]:
analyzer = PychometricsAnalyzer(
    api_key=api_key,
    model_name=llm_model,
)

# Choose your data layout: "csv" (one document per row) or "interview" (one .txt file per document).
data_type = "csv"

if data_type == "csv":
    results_llm, metadata_llm, errors_llm = analyzer.analyze_csv(
        csv_path=csv_path,
        text_column=text_column,
        subreddit_column=subreddit_column or None,
        author_column=author_column or None,
        codebook_path=codebook_path,
        output_dir="LLM_coding",  # timestamp suffix added automatically
    )
elif data_type == "interview":
    results_llm, metadata_llm, errors_llm = analyzer.analyze_directory(
        input_dir=input_dir,
        codebook_path=codebook_path,
        output_dir="LLM_coding",  # timestamp suffix added automatically
    )
else:
    raise ValueError("data_type must be 'csv' or 'interview'.")

The analyzer always returns and saves three dictionaries:
- `results` — what the LLM actually coded
- `metadata` — information about how the LLM was prompted (tokens, latency, model)
- `errors` — information on any calls that failed

These will be saved as JSON files inside the timestamped output directory.

In [ ]:
format_coding_table(results_llm)

### Retry Failed Documents (optional)

If any documents failed — due to API errors, rate limits, or malformed responses — you can retry only the failed ones without re-processing documents that already succeeded. Uncomment the code below and update `output_dir` to point to the timestamped directory from your run above.

In [ ]:
# recovered_results, recovered_meta, remaining_errors = analyzer.retry_errors(
#     output_dir="path/to/LLM_coding_YYYY-MM-DD_HHMMSS",
#     codebook_path=codebook_path,
# )

## Step 2 (Dummy): Stand-in Human Coding via a Second LLM Run

For the comparison step we need a second set of codings. In Part 1 we generate that by running the LLM a second time over the same documents — the variability between runs gives us something non-trivial to compare against.

Variables in this section are prefixed with `dummy_human_` to make it clear this is *not* real human data.

> If you have real human codings, **skip Part 1 entirely and jump to Part 2**, which shows the typical workflow.

In [ ]:
# Use the same analyzer/model — the second run produces our stand-in "human" data.
if data_type == "csv":
    dummy_human_results, dummy_human_metadata, dummy_human_errors = analyzer.analyze_csv(
        csv_path=csv_path,
        text_column=text_column,
        subreddit_column=subreddit_column or None,
        author_column=author_column or None,
        codebook_path=codebook_path,
        output_dir="dummy_human_coding",  # timestamp suffix added automatically
    )
elif data_type == "interview":
    dummy_human_results, dummy_human_metadata, dummy_human_errors = analyzer.analyze_directory(
        input_dir=input_dir,
        codebook_path=codebook_path,
        output_dir="dummy_human_coding",  # timestamp suffix added automatically
    )
else:
    raise ValueError("data_type must be 'csv' or 'interview'.")

In [ ]:
format_coding_table(dummy_human_results)

## Step 3 (Dummy): Comparison

This step aligns the dummy-human and LLM codings for each document. For each construct, an LLM matcher reads all quotes from both coders and decides which dummy-human quotes and LLM quotes refer to the same passage or idea — even if the wording differs (paraphrase).

Each instance in the resulting DataFrame is classified as one of three statuses:
- **`matched`** — both coders identified this passage (True Positive, TP). Includes a `match_confidence` score from the matcher and a `span_overlap` score (Jaccard overlap of character indices).
- **`human_only`** — the (dummy) human identified this passage but the LLM missed it (False Negative, FN).
- **`llm_only`** — the LLM identified this passage but the (dummy) human missed it (False Positive, FP).

You can optionally use a different model for the matching step than for coding — for example, a more capable model for matching if accuracy is critical. Set `match_model` accordingly.

In [ ]:
comparator = PychometricsComparer(
    api_key=api_key,
    match_model=llm_model,
)

comparison_results = comparator.compare_results(
    human_results=dummy_human_results,
    llm_results=results_llm,
    output_dir="dummy_comparison_run",  # optional; saves JSON comparison files
)

## Step 4 (Dummy): Build Summary Tables

Now that we have compared the LLM coding to the dummy-human coding, we can produce summary tables that make the results easier to read.

These tables answer slightly different questions:

- **Concatenated summary:** if we pool all coded examples across all documents together, how well did the LLM perform for each construct?
- **Per-document summary:** how well did the LLM perform within each individual document?
- **Weighted summary:** what does performance look like across documents when documents with more coded material count more heavily?

### Important note about the metrics

This is an open-ended coding task, not a standard classification task with true negatives.

We observe:
- True positives (TP)
- False positives (FP)
- False negatives (FN)

But we do **NOT** have true negatives (TN), so:
- Specificity is not meaningful
- ROC AUC is not appropriate

**A note on Cohen's Kappa.** Cohen's Kappa adjusts agreement for chance, but it requires a true-negative count. Because open-ended span coding has no observable true negatives (TN = 0 by convention), kappa is biased downward and is not directly comparable to kappa values from fixed-item rating tasks. For this reason, kappa is **omitted from the summary tables by default**. We report **PABAK** (prevalence-adjusted bias-adjusted kappa) instead — it is more stable when prevalence is low and is a better fit for this kind of analysis. If you want kappa anyway for diagnostic purposes, pass `include_cohens_kappa=True` to `compute_summary_tables(...)`.

Reported metrics: precision, recall (sensitivity), F1, PABAK, and PR-AUC.

### 4.1 Row-Level Comparison DataFrame

The raw DataFrame below has one row per coded instance across all documents. Each row represents a single quote that was identified by at least one coder, with the full quote text, character indices, and TP/FP/FN indicators. This is the most granular view and is useful for inspecting specific mismatches — for example, filtering to `status == 'llm_only'` to review what the LLM found that the dummy human missed.

In [ ]:
comparison_table = format_comparison_table(comparison_results)
comparison_table

In [ ]:
per_interview, concatenated, weighted_summary = compute_summary_tables(comparison_table)

### 4.2 Concatenated Metrics (primary view)

One row per construct, with counts pooled across **all documents** before metrics are computed. Think of this as treating the entire dataset as a single document. This table answers: *overall, across all documents, how did the LLM perform on each construct?*

The `interviews_with_construct [p5-p95]` column shows how many documents contained at least one instance of the construct, along with the 5th and 95th percentile of instance counts — giving a sense of how common and variable the construct is across your data.

In [ ]:
format_concatenated(concatenated)

### Alternate Summary Tables

The two tables below offer complementary views to the concatenated summary above. Use them when you need to dig into individual-document performance or want results that down-weight documents with very little coded material.

#### Per-Document Metrics

One row per (document, construct) combination. Shows raw TP/FP/FN counts and all metrics broken down by document. Use this to identify whether performance issues are consistent across documents or driven by specific ones. Constructs that do not appear in a given document are absent from this table — they do not appear as zero rows. PR AUC is computed per (document, construct) where there are enough positive and negative predictions; otherwise it is `NaN`.

In [ ]:
format_per_interview(per_interview)

#### Weighted Summary

One row per construct. Metrics are first computed per document, then summarized as **weighted median [min–max]** across documents, where each document is weighted by its union size (TP + FP + FN) — so documents with more coded instances contribute more to the median. This table answers: *what does performance typically look like at the individual document level?* The `[min–max]` brackets show the range across documents, useful for spotting constructs with highly variable performance.

A `—` in the PR AUC column means there were insufficient label classes in every document to compute the metric for that construct.

In [ ]:
format_weighted_summary(weighted_summary)

---

# Part 2 — Real Human Data Workflow

In a typical research workflow you will compare the LLM's coding against **real human-coded data**, not against a second LLM run. This part walks through that workflow end-to-end.

The steps are the same as Part 1, with one key difference: instead of generating a stand-in human coding via a second LLM run, you load human-coded data from a CSV or xlsx file (e.g., a Dedoose excerpt export, or any other tabular human-coding format) using `load_human_coding(...)`.

## Step 1 (Real): LLM Coding

This is the same LLM-coding step as in Part 1.

> **If you already ran the LLM in Part 1 and want to reuse those results**, skip this cell and continue to Step 2 below uses `results_llm` from above and your `comparator` is already configured.
>
> **Run this cell only if you want a fresh, independent LLM run for Part 2** (e.g., a different model, a different output directory, or you skipped Part 1 entirely).

In [ ]:
# Run this only if you want a fresh LLM run for Part 2; otherwise skip and reuse `results_llm` from Part 1.

real_analyzer = PychometricsAnalyzer(
    api_key=api_key,
    model_name=llm_model,
)

if data_type == "csv":
    results_llm, metadata_llm, errors_llm = real_analyzer.analyze_csv(
        csv_path=csv_path,
        text_column=text_column,
        subreddit_column=subreddit_column or None,
        author_column=author_column or None,
        codebook_path=codebook_path,
        output_dir="LLM_coding_real",  # timestamp suffix added automatically
    )
elif data_type == "interview":
    results_llm, metadata_llm, errors_llm = real_analyzer.analyze_directory(
        input_dir=input_dir,
        codebook_path=codebook_path,
        output_dir="LLM_coding_real",  # timestamp suffix added automatically
    )
else:
    raise ValueError("data_type must be 'csv' or 'interview'.")

format_coding_table(results_llm)

## Step 2 (Real): Loading Human-Coded Data

Real human coding typically lives in a tabular file (CSV, TSV, or xlsx) where each row is a single coded excerpt. `load_human_coding(...)` reads that file and converts it into the same internal `AnalysisResult` format that `analyze_csv` / `analyze_directory` produce, so it can be passed straight to the comparator.

The loader is **generic** — it works with any tabular human-coding source as long as you tell it which columns to use:

| Argument            | Default                       | What it identifies                                                    |
|---------------------|-------------------------------|------------------------------------------------------------------------|
| `doc_id_col`        | `"Media Title"`               | Column whose value matches the LLM `document_id` (filename / CSV row) |
| `quote_col`         | `"Excerpt Copy"`              | The verbatim text excerpt the human selected                           |
| `range_col`         | `"Excerpt Range"`             | Character-level start–end indices into the source document             |
| `construct_col`     | `"Codes Applied Combined"`    | Code(s) applied to the excerpt; multiple codes are split on the separator |
| `range_format`      | `"dash"` (e.g. `"858-1159"`)  | `"dash"` for Dedoose-style; `"colon"` for `start:end` format           |
| `construct_separator` | `","`                       | Used to split multi-code cells into individual constructs              |

The defaults match a Dedoose **Excerpt Export** out of the box, so a Dedoose user only needs `load_human_coding(path)`. For other tabular sources, override the column-name kwargs accordingly.

> **Important — document-id alignment.** The values in `doc_id_col` must match the document IDs used in your LLM run (typically the source filename without extension, or the CSV-row identifier built from `subreddit_column` + `author_column`). If they don't align, the comparator won't pair instances correctly.

In [ ]:
# Generic example — override only the kwargs that differ from the defaults.
# Below is the Dedoose-default call (no overrides). For a non-Dedoose source, pass the
# column names explicitly, e.g.:
#
# human_results = load_human_coding(
#     human_csv_path,
#     doc_id_col="participant_id",
#     quote_col="excerpt_text",
#     range_col="char_range",
#     construct_col="code",
#     range_format="colon",          # if your ranges look like "858:1159"
#     construct_separator=";",        # if codes are separated by something else
# )

human_results = load_human_coding(human_csv_path)

### Optional: Validate human-coded constructs against the codebook

Loading does **not** validate that the constructs found in the human file all exist in your codebook — that's deliberate, so you can inspect the file first. Run the cell below if you want a sanity check before comparison; it returns a `ValidationReport` that lists any unknown constructs by document. With `strict=True` it raises an error if any unknown constructs are found.

In [ ]:
validation_report = validate_against_codebook(
    results=human_results,
    codebook=codebook_path,
    strict=False,
)
print(validation_report)

## Step 3 (Real): Comparison

Same comparison step as Part 1, but now we pass the real human results in place of the dummy ones.

In [ ]:
real_comparator = PychometricsComparer(
    api_key=api_key,
    match_model=llm_model,
)

comparison_results_real = real_comparator.compare_results(
    human_results=human_results,
    llm_results=results_llm,
    output_dir="real_comparison_run",  # optional; saves JSON comparison files
)

## Step 4 (Real): Summary Tables

Same four summary tables as Part 1, in the same order: row-level table → concatenated → alternate (per-document + weighted). See the metrics note in Part 1 Step 4 for details on PABAK vs Cohen's Kappa.

### 4.1 Row-Level Comparison DataFrame

One row per coded instance across all documents — useful for inspecting specific mismatches.

In [ ]:
comparison_table_real = format_comparison_table(comparison_results_real)
comparison_table_real

In [ ]:
per_interview_real, concatenated_real, weighted_summary_real = compute_summary_tables(comparison_table_real)

### 4.2 Concatenated Metrics (primary view)

Counts pooled across all documents — overall LLM performance per construct against your real human coders.

In [ ]:
format_concatenated(concatenated_real)

### Alternate Summary Tables

#### Per-Document Metrics

Performance broken down by document — useful for spotting documents that drive overall numbers.

In [ ]:
format_per_interview(per_interview_real)

#### Weighted Summary

Per-document metrics summarized as weighted median [min–max] across documents.

In [ ]:
format_weighted_summary(weighted_summary_real)